# Practical 09: Missing Value Imputation with SimpleImputer & Categorical Encoding

**UNIT 3: Preparing Big Data for Machine Learning**

**Problem Statement:** Write a Python program using Scikit-Learn to preprocess a machine learning dataset: handle missing numerical values using SimpleImputer with the mean strategy, transform categorical text labels into ordinal integers using LabelEncoder, and convert nominal variables to binary dummy indicators using One-Hot Encoding.

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

rg = np.random.default_rng(99)
n = 55
age = rg.integers(23, 61, n).astype(float)
salary = np.round(28500 + age * 970 + rg.normal(0, 7000, n), -2)
frame = pd.DataFrame({
    "Country": rg.choice(["Norway", "Chile", "Kenya"], n),
    "Age": age,
    "Salary": salary,
    "Purchased": np.where(salary + rg.normal(0, 8000, n) > 55000, "Yes", "No"),
})
frame.loc[rg.choice(n, 5, replace=False), "Age"] = np.nan
frame.loc[rg.choice(n, 5, replace=False), "Salary"] = np.nan
print(frame.head(11))

print(frame.isna().sum())

si = SimpleImputer(missing_values=np.nan, strategy="mean")
frame[["Age", "Salary"]] = si.fit_transform(frame[["Age", "Salary"]])

print("Learned means (Age, Salary):", si.statistics_.round(2))
print("\nMissing values after imputation:")
print(frame.isna().sum())

le_target = LabelEncoder()
frame["Purchased_Encoded"] = le_target.fit_transform(frame["Purchased"])
print("Purchased mapping:", dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

frame["Country_Label"] = LabelEncoder().fit_transform(frame["Country"])
print("\nNote: Country_Label implies an artificial order (Chile < Kenya < Norway),")
print("so LabelEncoder is unsuitable for nominal features in linear models.")
print(frame[["Country", "Country_Label", "Purchased", "Purchased_Encoded"]].head())

hot = OneHotEncoder(sparse_output=False, dtype=int)
encoded = hot.fit_transform(frame[["Country"]])
ohe_df = pd.DataFrame(encoded, columns=hot.get_feature_names_out(["Country"]), index=frame.index)

encoded_df = pd.concat([frame.drop(columns=["Country", "Country_Label"]), ohe_df], axis=1)
print(encoded_df.head(11))

print(pd.get_dummies(frame["Country"], prefix="Country", dtype=int).head())

,Country,Age,Salary,Purchased
0,Chile,59.0,87000.0,Yes
1,Norway,42.0,68700.0,No
2,Kenya,51.0,84700.0,Yes
3,Kenya,44.0,76400.0,Yes
4,Chile,29.0,56300.0,Yes
5,Chile,42.0,64700.0,Yes
6,Kenya,58.0,98500.0,Yes
7,Norway,59.0,90600.0,Yes
8,Chile,59.0,74700.0,Yes
9,Chile,46.0,79000.0,Yes


Country      0
Age          5
Salary       5
Purchased    0
dtype: int64
Learned means (Age, Salary): [4.3320e+01 7.0026e+04]

Missing values after imputation:
Country      0
Age          0
Salary       0
Purchased    0
dtype: int64
Purchased mapping: {'No': np.int64(0), 'Yes': np.int64(1)}

Note: Country_Label implies an artificial order (Chile < Kenya < Norway),
so LabelEncoder is unsuitable for nominal features in linear models.

,Country,Country_Label,Purchased,Purchased_Encoded
0,Chile,0,Yes,1
1,Norway,2,No,0
2,Kenya,1,Yes,1
3,Kenya,1,Yes,1
4,Chile,0,Yes,1


,Age,Salary,Purchased,Purchased_Encoded,Country_Chile,Country_Kenya,Country_Norway
0,59.0,87000.0,Yes,1,1,0,0
1,42.0,68700.0,No,0,0,0,1
2,51.0,84700.0,Yes,1,0,1,0
3,44.0,76400.0,Yes,1,0,1,0
4,29.0,56300.0,Yes,1,1,0,0
5,42.0,64700.0,Yes,1,1,0,0
6,58.0,98500.0,Yes,1,0,1,0
7,59.0,90600.0,Yes,1,0,0,1
8,59.0,74700.0,Yes,1,1,0,0
9,46.0,79000.0,Yes,1,1,0,0


,Country_Chile,Country_Kenya,Country_Norway
0,1,0,0
1,0,0,1
2,0,1,0
3,0,1,0
4,1,0,0
